# Sección 10. Interpretación, limitaciones y conclusiones

Esta sección **no calcula** resultados nuevos: integra los de las Secciones 3 a 9 —cargando
las tablas ya generadas— y los interpreta en conjunto. Se distingue explícitamente entre
**descripción** (lo que se observa en la muestra), **asociación** (patrones entre variables) e
**inferencia** (afirmaciones sobre la población), y se cierra con las limitaciones y las
conclusiones integradas.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import pandas as pd
from IPython.display import Markdown, display

TABLAS = REPO / "outputs" / "tablas"

resumen_global = pd.read_csv(TABLAS / "eda_resumen_global.csv")
topologia = pd.read_csv(TABLAS / "topologia.csv")
comunidades_resumen = pd.read_csv(TABLAS / "comunidades_resumen.csv")
sent = pd.read_csv(TABLAS / "sentimiento_comentarios.csv")
sent_canal = pd.read_csv(TABLAS / "sentimiento_por_canal.csv")
sent_com = pd.read_csv(TABLAS / "sentimiento_por_comunidad.csv")
print('Tablas de resultados cargadas para la sintesis.')

Tablas de resultados cargadas para la sintesis.


## 10.1. Hallazgos en el contexto de participación y consumo en YouTube

**La participación es escasa y muy concentrada.** De 293 videos recolectados, solo **19
(6.5 %)** tienen al menos un comentario, y 406 comentarios provienen de 332 autores. La
co-participación forma una red **fragmentada** (10 componentes; la mayor concentra el 81 % de
los nodos en la bipartita) y **dispersa**: la proyección autor–autor tiene densidad ≈ 0.20
pero una transitividad de 0.98, señal de que la mayoría de conexiones vienen de *compartir un
mismo video* (cliques por video) más que de tender puentes entre videos.

**Pocas cuentas y videos sostienen la estructura.** Unos pocos autores comentan en varios
videos y actúan como **puente** (Sec. 8): si se eliminaran, la red se fragmentaría más. La
participación se concentra en un puñado de videos de alto tráfico (p. ej. *Qué rico come tu
diputado*, con 159–161 comentarios).

**El tono es mayoritariamente negativo, pero depende del contenido.** El 61 % de los
comentarios es negativo (score medio −0.38). El único grupo netamente positivo es contenido
**promocional** (Municipalidad de Guatemala, *Plan 2032*: +0.61), frente a la fuerte
negatividad de los videos de **denuncia política**. Sentimiento y comunidad coinciden cuando
la comunidad se organiza alrededor de un video de tono homogéneo.

In [2]:
# Síntesis cuantitativa que respalda 10.1 (todo proviene de tablas ya calculadas).
tabla = pd.DataFrame([
    ('Videos recolectados', '293'),
    ('Videos con >=1 comentario', '19 (6.5%)'),
    ('Comentarios / autores', '406 / 332'),
    ('Componentes (bipartita) / % mayor', '10 / 81%'),
    ('Densidad proy. autor-autor', f"{topologia.loc[topologia.red=='autor-autor','densidad'].iloc[0]:.3f}"),
    ('Transitividad proy. autor-autor', f"{topologia.loc[topologia.red=='autor-autor','transitividad'].iloc[0]:.3f}"),
    ('Modularidad (comunidades)', f"{comunidades_resumen['modularidad_particion'].iloc[0]:.3f}"),
    ('Comentarios negativos', f"{(sent.etiqueta=='negativo').mean():.0%}"),
    ('Score de sentimiento medio', f"{sent.score.mean():.2f}"),
], columns=['indicador', 'valor'])
display(tabla)

,indicador,valor
0,Videos recolectados,293
1,Videos con >=1 comentario,19 (6.5%)
2,Comentarios / autores,406 / 332
3,Componentes (bipartita) / % mayor,10 / 81%
4,Densidad proy. autor-autor,0.195
5,Transitividad proy. autor-autor,0.984
6,Modularidad (comunidades),0.395
7,Comentarios negativos,61%
8,Score de sentimiento medio,-0.38


## 10.2. Limitaciones

1. **Cobertura de comentarios.** Solo 19 de 293 videos tienen comentarios y la mayoría se
   concentra en unos pocos. La red y el sentimiento describen **esa** participación observada,
   no la actividad real de cada video ni del canal.
2. **Selección por consultas de búsqueda.** Los videos se obtuvieron mediante consultas
   (`source_query`) sobre política y noticias de Guatemala. La muestra está **sesgada por ese
   procedimiento**: el predominio de tono negativo refleja en parte *qué* se buscó.
3. **Fechas relativas.** `published_time` es relativo al momento de recolección ("hace 2
   días"); no permite un eje temporal exacto ni análisis de evolución.
4. **Conteos observados al momento de recolección.** Vistas, likes y `reply_count` son una
   foto puntual; pueden haber cambiado después.
5. **Falta de relaciones explícitas entre autores.** `reply_count` no identifica autores de
   respuestas, así que **no existe** una arista usuario→usuario real. La red es de
   co-participación, no de conversación.
6. **Concentración en pocos videos.** Muchos grupos (videos, comunidades) quedan por debajo
   del umbral de muestra y no se interpretan.
7. **El sentimiento mide tono del texto, no postura.** El modelo es fuerte para español de
   redes, pero no detecta ironía de forma confiable ni distingue crítica de aprobación de una
   acción (celebrar que 'caiga preso un corrupto' se etiqueta negativo).

## 10.3. Descripción, asociación e inferencia

| Nivel | Qué permite decir | Ejemplo de este trabajo |
|---|---|---|
| **Descripción** | Resumir la muestra observada | "En la muestra, el 61 % de los comentarios es negativo." |
| **Asociación** | Relacionar variables sin causa | "Los videos de denuncia se asocian con tono más negativo que los promocionales." |
| **Inferencia** | Generalizar a la población | *No sostenida por estos datos.* |

Todos los resultados de este laboratorio son **descriptivos y asociativos sobre la muestra
recolectada**. No se generaliza a todos los usuarios de YouTube ni a la población de Guatemala:
la selección por consultas y la baja cobertura lo impiden. Tampoco se afirma causalidad (p. ej.,
que un canal 'genere' negatividad); solo se describen asociaciones observadas.

## 10.4. Conclusiones integradas

- **Red.** La participación observada forma una red **fragmentada y concentrada**: pocos videos
  y pocas cuentas puente sostienen la conexión; la mayoría de autores solo aparece en un video.
  La estructura es de co-participación, no de conversación entre usuarios.
- **Comunidades.** Louvain recupera comunidades con modularidad ≈ 0.39, y las principales se
  alinean con **videos/canales concretos** más que con temas transversales: la gente se agrupa
  alrededor del contenido que comenta.
- **Contenido y sentimiento.** El tono dominante es negativo (61 %), consistente con un corpus
  de política y noticias; la excepción positiva es contenido promocional. Sentimiento y
  comunidad **coinciden** cuando la comunidad gira en torno a un video de tono homogéneo, lo que
  conecta el análisis de redes con el de contenido.
- **Lectura conjunta.** Estructura (redes), organización (comunidades) y tono (sentimiento)
  apuntan a lo mismo: la participación observada está **dominada por unos pocos videos de alto
  tráfico y tono crítico**, con audiencias que se agrupan por video. Es una descripción de la
  muestra recolectada, condicionada por el procedimiento de búsqueda, no un retrato de YouTube
  ni de Guatemala.

In [3]:
# --- Auto-verificación ------------------------------------------------------
assert (sent.etiqueta == 'negativo').mean() > 0.5
assert topologia.loc[topologia.red=='autor-autor','transitividad'].iloc[0] > 0.9
assert (sent_canal.query('canal.str.contains("Municipalidad")')['score_medio'] > 0).all()
assert comunidades_resumen['modularidad_particion'].iloc[0] > 0.3
print('Etapa 10: self-check OK — sintesis coherente con las tablas de las Secciones 3-9')

Etapa 10: self-check OK — sintesis coherente con las tablas de las Secciones 3-9
